# Probe学習文数比較 v2 — 保存済み特徴量から再開

v1のDataFrame.style属性衝突を修正。v1の960説明文と120 NPZを固定コミットから読み込み、学習・評価だけ実施する。OpenAI API・Qwen再推論・GPU・Drive不要。CPUランタイム、Colab Secretsは GITHUB_RESULTS_TOKEN のみ。旧結果は保持。v2専用結果ディレクトリに保存。

In [ ]:
%pip -q install pandas scikit-learn matplotlib requests tqdm

In [ ]:
import io, json, re, hashlib, base64, time, unicodedata, random
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from google.colab import userdata
from IPython.display import display

REPO="ryouy/GraduationThesis"
SOURCE_REF="328e2c15a0ec64bbbd68112edf95ec7dd31485e3"
SOURCE_BLOB="c0eebc8cbc5b292cab1cc4f7587fb617f16c60b7"
SOURCE="experiments/2026-11/1124_probe_50pairs/results/20261124_probe50_balanced_v5/pair_probe_scores.csv"
RUN_ID="20261126_probe_learning_curve_v2"
DEST="experiments/2026-11/1126_probe_learning_curve/results/"+RUN_ID
MODEL="Qwen/Qwen2.5-0.5B-Instruct"
REV="7ae557604adf67be50417f59c2c2f167def9a775"
API_MODEL="gpt-6-luna"
REASONING_EFFORT="medium"
PAIR_IDS=["C01","C27","C35","C23","C37"]
STYLES=["structure","action","situation","function"]
SIZES=[8,16,32,64]
SEEDS=[42,101,211,307,503]
TRAIN_PER_STYLE=16
EVAL_PER_STYLE=8
GEN_BATCH=8
MAX_ATTEMPTS=8
MAX_API_CALLS=500
FEATURE_CHUNK=8
LAYERS=list(range(1,25))
C=0.1
RUN_API=False
WORK=Path("/content/probe_learning_curve")/RUN_ID
WORK.mkdir(parents=True,exist_ok=True)

def secret(name):
    v=userdata.get(name)
    if not v: raise RuntimeError("Set Colab Secret "+name)
    return v
gh=requests.Session()
gh.headers.update({"Authorization":"Bearer "+secret("GITHUB_RESULTS_TOKEN"),"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"})
GH=f"https://api.github.com/repos/{REPO}/contents/"
def sha(b): return hashlib.sha256(b).hexdigest()
def blob(b): return hashlib.sha1(b"blob "+str(len(b)).encode()+b"\0"+b).hexdigest()
def dump(o): return (json.dumps(o,sort_keys=True,ensure_ascii=False,indent=2,allow_nan=False)+"\n").encode()
def meta(path,ref="main"):
    r=gh.get(GH+path,params={"ref":ref},timeout=90)
    if r.status_code==404:return None
    r.raise_for_status();return r.json()
def get(path,ref="main"):
    m=meta(path,ref)
    if m is None:return None
    if m.get("encoding")=="base64":b=base64.b64decode(m["content"])
    else:
        r=gh.get(GH+path,params={"ref":ref},headers={"Accept":"application/vnd.github.raw+json"},timeout=120)
        r.raise_for_status();b=r.content
    if blob(b)!=m["sha"]:raise RuntimeError("Git blob mismatch "+path)
    return b
def put(path,b):
    # Retry only GitHub writes; never overwrite existing artifacts.
    for attempt in range(8):
        m=meta(path)
        if m is not None:
            if m.get("sha")!=blob(b):raise RuntimeError("Immutable conflict "+path)
            return
        try:
            r=gh.put(GH+path,json={"branch":"main","message":"Probe learning curve v2: "+path.rsplit("/",1)[-1],"content":base64.b64encode(b).decode()},timeout=120)
            r.raise_for_status()
            time.sleep(1.2)
            return
        except requests.RequestException as e:
            status=getattr(e.response,"status_code",None)
            if status not in (None,409,429,500,502,503,504) or attempt==7:raise
            time.sleep(min(60,2**(attempt+1)))
def save(name,b):
    put(DEST+"/"+name,b)
    p=WORK/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(b)
def save_df(name,df):save(name,df.to_csv(index=False).encode())
raw=get(SOURCE,SOURCE_REF)
assert raw is not None and blob(raw)==SOURCE_BLOB
all_pairs=pd.read_csv(io.BytesIO(raw))
pairs=all_pairs.set_index("candidate_id").loc[PAIR_IDS].reset_index()
names=sorted(set(pairs.target_x)|set(pairs.comparator_y))
assert len(names)==10 and pairs.category.nunique()==5
display(pairs[["candidate_id","category","target_x","comparator_y","L24_auc"]])
partners={n:set() for n in names}
for r in pairs.itertuples():
    partners[r.target_x].add(r.comparator_y);partners[r.comparator_y].add(r.target_x)
save_df("selected_pairs.csv",pairs)
print("Expected sentences:",len(names)*(TRAIN_PER_STYLE+EVAL_PER_STYLE)*len(STYLES))

## 1. 固定済みデータと特徴量の復元
元のファイルのGit blob・文章CSVのSHA256・NPZの開始終了位置・層・形状を検証する。

In [ ]:
PARENT="experiments/2026-11/1126_probe_learning_curve/results/20261126_probe_learning_curve_v1"
PARENT_REF="ddf3d330065399738e2322eaabf03f8790a0d39b"
PARENT_DESC_BLOB="ebd3d8778950954f09f7688ad1f3ec1f5c51eab2"
PARENT_CONFIG_BLOB="b0207cd64df080dc93811a33f202bc3c969072b3"
parent_raw=get(PARENT+"/config.json",PARENT_REF)
assert blob(parent_raw)==PARENT_CONFIG_BLOB
config=json.loads(parent_raw)
assert config["pairs"]==PAIR_IDS and config["layers"]==LAYERS and config["sizes"]==SIZES and config["seeds"]==SEEDS and config["probe_C"]==C
config.update({"parent_ref":PARENT_REF,"parent_run":"20261126_probe_learning_curve_v1","parent_config_blob":PARENT_CONFIG_BLOB,"parent_description_blob":PARENT_DESC_BLOB,"run_id":RUN_ID,"repair":"explicit DataFrame column indexing; no API or model inference"})
save("config.json",dump(config))
text_raw=get(PARENT+"/descriptions.csv",PARENT_REF)
assert blob(text_raw)==PARENT_DESC_BLOB
texts=pd.read_csv(io.BytesIO(text_raw),keep_default_na=False)
assert len(texts)==960 and texts["text_id"].is_unique
assert set(texts["concept"])==set(names)
assert set(texts["style"])==set(STYLES)
assert set(texts["split"])=={"train","eval"}
counts=texts.groupby(["concept","split","style"]).size()
assert len(counts)==80 and (counts.xs("train",level="split")==16).all() and (counts.xs("eval",level="split")==8).all()
text_hash=sha(text_raw)
parts=[]
for start in tqdm(range(0,len(texts),FEATURE_CHUNK),desc="Restore v1 features"):
    end=min(start+FEATURE_CHUNK,len(texts))
    name=f"features/chunk_{start:04d}.npz"
    b=get(PARENT+"/"+name,PARENT_REF)
    assert b is not None,name
    with np.load(io.BytesIO(b),allow_pickle=False) as z:
        assert int(z["start"])==start and int(z["end"])==end and str(z["text_hash"])==text_hash and z["layers"].tolist()==LAYERS
        a=z["features"].copy()
        assert a.shape==(end-start,24,896) and np.isfinite(a).all()
    parts.append(a)
V=np.concatenate(parts)
assert V.shape==(960,24,896)
print("Verified:",V.shape)
display(counts.unstack("split"))

## 3. 入れ子学習曲線

各seedで文章タイプごとに学習文の順番を固定し、先頭2/4/8/16文を使用する。評価用32文/ラベルは共通。64文では全学習データを使うのでseed間の反復は実質同一（1回のみ）。StandardScalerは学習データのみにfit。評価ROC-AUCと各文章タイプのAUCを全層保存する。

In [ ]:
rows=[]; predictions=[]
for p in tqdm(list(pairs.itertuples()),desc="Pairs"):
    x,y=p.target_x,p.comparator_y
    ev=[texts.index[(texts["concept"]==n)&(texts["split"]=="eval")].to_numpy() for n in [x,y]]
    test=np.concatenate(ev); yt=np.r_[np.ones(len(ev[0]),dtype=int),np.zeros(len(ev[1]),dtype=int)]
    assert len(test)==64
    for seed in SEEDS:
        rng=np.random.default_rng(seed)
        orders={}
        for n in [x,y]:
            for style in STYLES:
                inds=texts.index[(texts["concept"]==n)&(texts["split"]=="train")&(texts["style"]==style)].to_numpy()
                assert len(inds)==TRAIN_PER_STYLE,(n,style,len(inds))
                orders[n,style]=rng.permutation(inds)
        for ntrain in SIZES:
            if ntrain==64 and seed!=SEEDS[0]:continue
            train=np.concatenate([orders[n,s][:ntrain//4] for n in [x,y] for s in STYLES])
            ytr=np.r_[np.ones(ntrain,dtype=int),np.zeros(ntrain,dtype=int)]
            assert len(train)==2*ntrain and len(ytr)==len(train) and len(set(train))==len(train)
            assert not set(train)&set(test)
            for li,l in enumerate(LAYERS):
                sc=StandardScaler().fit(V[train,li])
                clf=LogisticRegression(C=C,solver="liblinear",random_state=42,max_iter=2000).fit(sc.transform(V[train,li]),ytr)
                score=clf.decision_function(sc.transform(V[test,li]))
                row={"candidate_id":p.candidate_id,"target_x":x,"comparator_y":y,"layer":l,"n_train_per_label":ntrain,"seed":seed,"auc":roc_auc_score(yt,score)}
                for style in STYLES:
                    mask=texts["style"].iloc[test].to_numpy()==style
                    row["auc_"+style]=roc_auc_score(yt[mask],score[mask])
                rows.append(row)
                if l==24:
                    predictions.extend([{**{k:row[k] for k in ["candidate_id","n_train_per_label","seed"]},"text_id":texts.text_id.iloc[j],"label":int(label),"score":float(z)} for j,label,z in zip(test,yt,score)])
scores=pd.DataFrame(rows); assert len(scores)==5*(5*3+1)*24
save_df("learning_curve_scores.csv",scores)
save_df("l24_eval_predictions.csv",pd.DataFrame(predictions))
summary=scores.groupby(["candidate_id","target_x","layer","n_train_per_label"]).auc.agg(["mean","std","min","max","count"]).reset_index()
save_df("learning_curve_summary.csv",summary)
display(summary[summary.layer==24])

## 4. 比較と暫定判断

32文のL24平均AUCが64文より0.02以内の低下で、かつ0.05超の低下が5組中1組以下なら32文を推奨する。それ以外は64文を候補とする。ただし、5組の小規模な探索結果であり、データ量効果・生成モデルの優劣の確証ではない。8/16文の結果も示す。

In [ ]:
last=summary[summary.layer==24]
wide=last.pivot(index=["candidate_id","target_x"],columns="n_train_per_label",values="mean")
wide["gap_64_minus_32"]=wide[64]-wide[32]
gap=float(wide.gap_64_minus_32.mean()); bad=int((wide.gap_64_minus_32>.05).sum())
recommend=32 if gap<=.02 and bad<=1 else 64
display(wide)
save_df("l24_size_comparison.csv",wide.reset_index())
for title,frame in [("L24 learning curve",last),("Mean learning curve across pairs",last.groupby("n_train_per_label")["mean"].mean().reset_index())]:
    fig,ax=plt.subplots(figsize=(7,4))
    if "target_x" in frame:
        for name,g in frame.groupby("target_x"):
            ax.plot(g.n_train_per_label,g["mean"],marker="o",label=name)
        ax.legend()
    else:ax.plot(frame.n_train_per_label,frame["mean"],marker="o")
    ax.axhline(.8,ls="--",color="gray");ax.set(xlabel="Training texts per label",ylabel="ROC-AUC",ylim=(0,1.02),xticks=SIZES,title=title)
    fig.tight_layout(); buf=io.BytesIO();fig.savefig(buf,format="png",dpi=150);save(title.lower().replace(" ","_")+".png",buf.getvalue());plt.show()
fig,ax=plt.subplots(figsize=(8,4))
for n,g in summary.groupby("n_train_per_label"):
    a=g.groupby("layer")["mean"].mean();ax.plot(a.index,a.values,label=str(n))
ax.set(xlabel="Layer",ylabel="Mean ROC-AUC",ylim=(0,1.02));ax.legend(title="Texts per label");fig.tight_layout()
buf=io.BytesIO();fig.savefig(buf,format="png",dpi=150);save("layer_learning_curves.png",buf.getvalue());plt.show()
msg=f"""# Probe学習曲線 v2 — 探索的結果

- v1の固定済み特徴量を再利用。固定5組・960説明文・4学習サイズ・24層。
- 32対64のL24平均AUC差（64−32）: {gap:.4f}
- 0.05超の低下: {bad}/5組
- 暫定推奨: 各ラベル{recommend}学習文
- 共通評価セットは各ラベル32文。正式採択には新しい独立評価と事前固定した検定が必要。
- 人手事実・意味的重複監査は未実施。高AUCは思考量や抑制下の妥当性を保証しない。
"""
save("analysis_summary.md",msg.encode())
files={p.name:sha(p.read_bytes()) for p in WORK.iterdir() if p.is_file()}
save("manifest.json",dump({"run_id":RUN_ID,"source_blob":SOURCE_BLOB,"config_sha256":sha(dump(config)),"n_texts":len(texts),"n_score_rows":len(scores),"recommended_train_per_label":recommend,"human_audit":"pending","feature_chunks":len(parts),"parent_ref":PARENT_REF,"parent_description_blob":PARENT_DESC_BLOB,"files":files}))
print(msg)
print("Results:",f"https://github.com/{REPO}/tree/main/{DEST}")